<a href="https://colab.research.google.com/github/vickytoriag/Home-Work/blob/main/hw12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!apt-get update -qq
!apt-get install -y tshark

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libbcg729-0 libc-ares2 liblua5.2-0 libnl-genl-3-200 libpcap0.8 libsbc1
  libsmi2ldbl libspandsp2 libspeexdsp1 libwireshark-data libwireshark15
  libwiretap12 libwsutil13 wireshark-common
Suggested packages:
  snmp-mibs-downloader geoipupdate geoip-database geoip-database-extra
  libjs-leaflet libjs-leaflet.markercluster wireshark-doc
The following NEW packages will be installed:
  libbcg729-0 libc-ares2 liblua5.2-0 libnl-genl-3-200 libpcap0.8 libsbc1
  libsmi2ldbl libspandsp2 libspeexdsp1 libwireshark-data libwireshark15
  libwiretap12 libwsutil13 tshark wireshark-common
0 upgraded, 15 newly installed, 0 to remove and 104 not upgraded.
Need to get 23.

In [4]:
!pip -q install pyshark pandas matplotlib nest_asyncio

In [5]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import pyshark
import nest_asyncio

nest_asyncio.apply()

pcap_file = "dns.cap"

print("Файл:", pcap_file)

dns_rows = []

cap = pyshark.FileCapture(
    pcap_file,
    display_filter="dns",
    use_json=True,
    keep_packets=False
)

for pkt in cap:
    try:
        t = getattr(pkt, "sniff_time", None)

        dns_layer = pkt.dns
        domain = getattr(dns_layer, "qry_name", None)

        src_ip = None
        dst_ip = None
        if hasattr(pkt, "ip"):
            src_ip = pkt.ip.src
            dst_ip = pkt.ip.dst
        elif hasattr(pkt, "ipv6"):
            src_ip = pkt.ipv6.src
            dst_ip = pkt.ipv6.dst

        if domain:
            dns_rows.append({
                "time": pd.to_datetime(t) if t is not None else pd.NaT,
                "domain": str(domain),
                "src_ip": src_ip,
                "dst_ip": dst_ip
            })
    except:
        continue

cap.close()

df_dns = pd.DataFrame(dns_rows)

print("DNS записей найдено:", len(df_dns))
display(df_dns.head(10))

# Сохранение лога
df_dns.to_csv("dns_artifacts.csv", index=False)
print("CSV сохранен: dns_artifacts.csv")

# Минимальная визуализация
if len(df_dns) > 0:
    top_domains = df_dns["domain"].value_counts().head(10)

    plt.figure(figsize=(10, 5))
    top_domains.plot(kind="bar")
    plt.xlabel("domain")
    plt.ylabel("count")
    plt.title("Частота DNS доменов")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("DNS не обнаружены в этом дампе.")

Файл: dns.cap
DNS записей найдено: 0


""


CSV сохранен: dns_artifacts.csv
DNS не обнаружены в этом дампе.
